<div align="center">
<img src="https://poorit.in/image.png" alt="Poorit" width="40" style="vertical-align: middle;"> <b>AI SYSTEMS ENGINEERING 2</b>

## Unit 2: Multi-Model, Structured Outputs, and Guardrails

**CV Raman Global University, Bhubaneswar**
*AI Center of Excellence*

</div>

---

### What You'll Learn

In this notebook, you will:

1. **Use multiple LLM providers** — run the same agent on DeepSeek, Gemini, Groq, or OpenAI with one-line changes
2. **Get structured outputs** — use Pydantic output_type to guarantee response format
3. **Add input guardrails** — validate user input before it reaches your agent

## 1. Environment Setup

In [ ]:
!pip install -q "openai-agents[litellm]" pydantic

In [ ]:
import os
import asyncio
from getpass import getpass
from pydantic import BaseModel, Field
from IPython.display import display, Markdown
from agents import (
    Agent, Runner, trace, function_tool,
    input_guardrail, GuardrailFunctionOutput,
)
from agents.extensions.models.litellm_model import LitellmModel

print("Enter your API keys (press Enter to skip optional ones):\n")

openai_key = getpass("OpenAI API Key (required): ")
os.environ['OPENAI_API_KEY'] = openai_key

google_key = getpass("Google Gemini API Key (optional): ")
if google_key:
    os.environ['GEMINI_API_KEY'] = google_key

groq_key = getpass("Groq API Key (optional): ")
if groq_key:
    os.environ['GROQ_API_KEY'] = groq_key

---

## 2. The Problem — One Model Doesn't Fit All

Different LLMs have different strengths:

| Provider | Model | Best For | Relative Cost |
|---|---|---|---|
| OpenAI | gpt-4o-mini | General tasks, tool use | $$ |
| Google | gemini-2.0-flash | Fast responses, long context | $ |
| Groq | llama-3.3-70b | Ultra-fast inference | $ |

In a real system, you might want a **cheap, fast model** for planning, a **capable model** for writing, and a **specialized model** for code. But wiring up each provider manually is tedious:

```python
# OLD way — 4 lines per provider, different clients, different configs
from openai import AsyncOpenAI
deepseek_client = AsyncOpenAI(base_url="https://api.deepseek.com/v1", api_key=...)
model = OpenAIChatCompletionsModel(model="deepseek-chat", openai_client=deepseek_client)
```

The Agents SDK's `LitellmModel` wraps [LiteLLM](https://docs.litellm.ai/) to give you **one-line access to 100+ providers**:

```python
# NEW way — 1 line per provider, same pattern for all
model = LitellmModel(model="openai/gpt-4o-mini")
model = LitellmModel(model="gemini/gemini-2.0-flash")
model = LitellmModel(model="groq/llama-3.3-70b-versatile")
```

---

## 3. Multi-Model Agents with LiteLLM

In [ ]:
instructions = "You are a helpful assistant. Answer the user's question concisely in 2-3 sentences."

# OpenAI — same pattern as every other provider
openai_agent = Agent(
    name="OpenAI Agent",
    instructions=instructions,
    model=LitellmModel(model="openai/gpt-4o-mini"),
)

# Build agents for available providers
agents = [openai_agent]

if os.getenv('GEMINI_API_KEY'):
    gemini_agent = Agent(
        name="Gemini Agent",
        instructions=instructions,
        model=LitellmModel(model="gemini/gemini-2.0-flash"),
    )
    agents.append(gemini_agent)
    print("✓ Gemini agent configured")

if os.getenv('GROQ_API_KEY'):
    groq_agent = Agent(
        name="Groq/Llama Agent",
        instructions=instructions,
        model=LitellmModel(model="groq/llama-3.3-70b-versatile"),
    )
    agents.append(groq_agent)
    print("✓ Groq/Llama agent configured")

print(f"\nTotal agents: {len(agents)}")

In [ ]:
question = "What are the key benefits of AI-powered compliance tools for banks?"

with trace("Multi-model comparison"):
    tasks = [Runner.run(agent, question) for agent in agents]
    results = await asyncio.gather(*tasks, return_exceptions=True)

for agent, result in zip(agents, results):
    print(f"\n{'='*50}")
    print(f"{agent.name}:")
    print(f"{'='*50}")
    if isinstance(result, Exception):
        print(f"Error: {result}")
    else:
        print(result.final_output)

> **Tip:** `trace()` groups all agent activity under a single trace you can inspect in the OpenAI dashboard. We showed it here once — from now on we'll skip it to keep cells short, but you can always add it back for debugging.

---

## 4. Structured Outputs with output_type

In the previous notebook, we saw how `@function_tool` uses type hints to define tool schemas. The Agents SDK applies the same idea to **agent outputs**: give an agent a Pydantic `output_type`, and it's forced to return data matching that exact structure.

This is the SDK's version of what we learned with `client.beta.chat.completions.parse()` in Unit 1 — but integrated directly into the agent:

```
┌─────────────────┐        ┌─────────────────────┐
│  Agent(          │        │  result.final_output │
│    output_type=  │  ───>  │                     │
│    EmailDraft    │        │  .subject = "..."    │
│  )               │        │  .body = "..."       │
│                  │        │  .cta = "..."        │
└─────────────────┘        └─────────────────────┘
```

No JSON parsing, no regex extraction — you get a **typed Python object** directly.

In [ ]:
class EmailDraft(BaseModel):
    """Structured email output."""
    subject: str = Field(description="Email subject line")
    greeting: str = Field(description="Opening greeting")
    body: str = Field(description="Main email body")
    call_to_action: str = Field(description="What you want the reader to do")
    signature: str = Field(description="Sign-off and sender name")

structured_agent = Agent(
    name="Structured Email Writer",
    instructions="You write professional cold sales emails for TechAudit India (AI-powered compliance tools). Always provide a complete, well-structured email.",
    model=LitellmModel(model="openai/gpt-4o-mini"),
    output_type=EmailDraft
)

In [ ]:
result = await Runner.run(structured_agent, "Write a sales email to a fintech CEO about SOC2 compliance automation")
email = result.final_output

# Access fields directly — no parsing needed
print(f"Subject: {email.subject}")
print(f"\n{email.greeting}")
print(f"\n{email.body}")
print(f"\n{email.call_to_action}")
print(f"\n{email.signature}")

# It's a real Pydantic object
print(f"\nType: {type(email).__name__}")
print(f"As dict: {email.model_dump()}")

---

## 5. Input Guardrails — The Bouncer Pattern

Before a user's message reaches your agent, you might want to **check it first** — block prompt injections, filter inappropriate content, or enforce business rules.

Think of guardrails as a **bouncer at a club**:

```
User Message                              
     │                                    
     ▼                                    
┌──────────────┐                          
│   Guardrail  │──── ✗ Block ──> "Sorry, I can't help with that."
│   (Bouncer)  │                          
└──────┬───────┘                          
       │ ✓ Pass                           
       ▼                                  
┌──────────────┐                          
│    Agent     │──> Normal response       
└──────────────┘                          
```

The guardrail itself is an agent with a structured output (pass/fail). It runs **before** the main agent, and if it triggers, the main agent never sees the message.

### How It Works

1. Define a Pydantic model for the guardrail's verdict
2. Create a small agent that checks the input
3. Wrap it with `@input_guardrail`
4. Attach it to your main agent with `input_guardrails=[...]`

In [ ]:
class TopicCheckResult(BaseModel):
    """Result of checking if the message is on-topic."""
    is_off_topic: bool
    reason: str

guardrail_checker = Agent(
    name="Topic Checker",
    instructions="""Determine if the user's message is related to business, sales, or compliance topics.
If the message is about something completely unrelated (e.g., recipes, sports scores, personal advice), flag it as off-topic.""",
    output_type=TopicCheckResult,
    model=LitellmModel(model="openai/gpt-4o-mini"),
)

@input_guardrail
async def topic_guardrail(ctx, agent, message):
    """Block messages that are off-topic for our sales agent."""
    result = await Runner.run(guardrail_checker, message, context=ctx.context)
    check = result.final_output

    if check.is_off_topic:
        print(f"⚠️ Guardrail triggered: {check.reason}")

    return GuardrailFunctionOutput(
        output_info={"check": check.model_dump()},
        tripwire_triggered=check.is_off_topic
    )

In [ ]:
protected_agent = Agent(
    name="Sales Assistant",
    instructions="You are a sales assistant for TechAudit India. Help users with questions about compliance tools, pricing, and business solutions.",
    model=LitellmModel(model="openai/gpt-4o-mini"),
    input_guardrails=[topic_guardrail]
)

In [ ]:
result = await Runner.run(protected_agent, "What compliance frameworks do your tools support?")
print(result.final_output)

In [ ]:
try:
    result = await Runner.run(protected_agent, "What's a good recipe for butter chicken?")
    print(result.final_output)
except Exception as e:
    print(f"Blocked: {e}")

---

## 6. Key Takeaways

### Concept Map

```
Multi-Model (LitellmModel)
  └── One-line access to any provider
      └── model=LitellmModel(model="provider/model-name")

Structured Outputs (output_type)
  └── Agent returns a typed Pydantic object
      └── result.final_output.field_name

Input Guardrails (@input_guardrail)
  └── Check input before it reaches the agent
      └── GuardrailFunctionOutput(tripwire_triggered=True/False)
```

> **Note:** We used `trace()` once (Section 3) to show how to group runs for debugging. It's always available via `from agents import trace` — wrap any `Runner.run()` call with `with trace("label"):` when you need it.

### Quick Reference

| Concept | Code | When to Use |
|---|---|---|
| **LiteLLM model** | `LitellmModel(model="provider/model")` | Use any LLM provider |
| **Structured output** | `Agent(output_type=MyModel)` | Need typed, predictable responses |
| **Input guardrail** | `@input_guardrail` + `GuardrailFunctionOutput` | Validate/filter user input |

---

## 7. Exercises

### Exercise 1: Multi-Model Comparison (Beginner)
Add a DeepSeek agent using `LitellmModel(model="deepseek/deepseek-chat")` to the multi-model comparison in Section 3. Compare the output quality across all providers for a business question of your choice.

### Exercise 2: Structured Output (Beginner)
Create a `ProductReview` Pydantic model with fields: `product_name` (str), `rating` (int, 1–5), `pros` (list[str]), `cons` (list[str]), and `summary` (str). Create an agent with `output_type=ProductReview` and test it with: *"Review the iPhone 16 Pro for a tech blog."*

### Exercise 3: Custom Guardrail (Intermediate)
Create a guardrail that checks if the user's message contains a **word count request** (e.g., "write 5000 words"). If the requested word count exceeds 500, block the request with a helpful message suggesting a shorter output. Attach it to the structured email writer from Section 4.

---

**What's Next?** In Unit 3, we'll explore CrewAI — a multi-agent framework built around the concept of crews, roles, and tasks.

---

**Course Information:**
- **Institution:** CV Raman Global University, Bhubaneswar
- **Program:** AI Center of Excellence
- **Course:** AI Systems Engineering 2
- **Developed by:** [Poorit Technologies](https://poorit.in) - *Transform Graduates into Industry-Ready Professionals*

---